# SmarTask GA — Experiment Analysis\n\nThis notebook loads the results from `run_experiments.py` and generates all visualisations.\n\n**Structure:**\n1. Setup & load data\n2. Summary table\n3. Heatmap — mean fitness per crossover × mutation\n4. Boxplot — fitness distribution per configuration\n5. Convergence curves — best configuration (all runs overlaid)\n6. Bar chart — mean min coverage unmet per configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

RESULTS_CSV = "results/experiment_results.csv"
CONV_DIR    = "results/convergence"

df = pd.read_csv(RESULTS_CSV)

# Derived column: short label for each configuration
df["config"] = df["crossover_type"] + "\n" + df["mutation_type"]

print(f"Loaded {len(df)} runs across {df['config'].nunique()} configurations")
df.head()

## 1. Summary Table\n\nMean, best, and std of fitness and min coverage unmet across all runs per configuration.

In [ ]:
summary = df.groupby(["crossover_type", "mutation_type"]).agg(
    fitness_mean=("best_fitness",       "mean"),
    fitness_best=("best_fitness",       "max"),
    fitness_std= ("best_fitness",       "std"),
    min_unmet_mean=("min_coverage_unmet", "mean"),
    min_unmet_best=("min_coverage_unmet", "min"),
    min_unmet_std= ("min_coverage_unmet", "std"),
    runs=("run", "count"),
).round(1).sort_values("fitness_best", ascending=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
summary

## 2. Heatmap — Mean Fitness per Crossover × Mutation

In [ ]:
pivot = df.groupby(["crossover_type", "mutation_type"])["best_fitness"].mean().unstack()

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto")

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=20, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i, j]:.0f}", ha="center", va="center", fontsize=9)

plt.colorbar(im, ax=ax, label="Mean fitness (higher = better)")
ax.set_title("Mean Fitness — Crossover × Mutation")
plt.tight_layout()
plt.savefig("results/heatmap_fitness.png", dpi=150)
plt.show()

## 3. Boxplot — Fitness Distribution per Configuration

In [ ]:
configs  = df["config"].unique()
data     = [df[df["config"] == c]["best_fitness"].values for c in configs]

fig, ax = plt.subplots(figsize=(14, 5))
bp = ax.boxplot(data, patch_artist=True, medianprops=dict(color="black", linewidth=2))

for patch in bp["boxes"]:
    patch.set_facecolor("steelblue")
    patch.set_alpha(0.6)

ax.set_xticks(range(1, len(configs) + 1))
ax.set_xticklabels(configs, fontsize=8)
ax.set_ylabel("Best fitness")
ax.set_title("Fitness Distribution per Configuration (5 runs each)")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.0f}"))
plt.tight_layout()
plt.savefig("results/boxplot_fitness.png", dpi=150)
plt.show()

## 4. Convergence Curves — Best Configuration (all runs overlaid)\n\nChange `BEST_CX` and `BEST_MUT` to the winning configuration after reviewing the summary table.

In [ ]:
BEST_CX  = "nbts"          # update after reviewing summary table
BEST_MUT = "demand_guided"  # update after reviewing summary table

pattern = f"{BEST_CX}__{BEST_MUT}__run*.npy"
import glob as _glob
files = sorted(_glob.glob(os.path.join(CONV_DIR, pattern)))

fig, ax = plt.subplots(figsize=(10, 4))
for f in files:
    curve = np.load(f)
    run_label = os.path.basename(f).split("__run")[-1].replace(".npy", "")
    ax.plot(curve, linewidth=1, alpha=0.7, label=f"Run {run_label}")

ax.set_xlabel("Generation")
ax.set_ylabel("Best fitness")
ax.set_title(f"Convergence Curves — {BEST_CX} + {BEST_MUT} (all runs)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("results/convergence_curves.png", dpi=150)
plt.show()

## 5. Bar Chart — Mean Min Coverage Unmet per Configuration

In [ ]:
bar_data = df.groupby("config")["min_coverage_unmet"].agg(["mean", "std"]).reset_index()
bar_data = bar_data.sort_values("mean")

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(bar_data["config"], bar_data["mean"],
              yerr=bar_data["std"], capsize=4,
              color="steelblue", alpha=0.7, edgecolor="black", linewidth=0.5)

ax.axhline(16, color="green", linestyle="--", linewidth=1.5, label="Optimal (16)")
ax.set_ylabel("Mean min coverage unmet (worker-days)")
ax.set_title("Coverage Gap per Configuration — lower is better")
ax.set_xticklabels(bar_data["config"], fontsize=8)
ax.legend()
plt.tight_layout()
plt.savefig("results/barchart_coverage.png", dpi=150)
plt.show()